# 06 - Gold: Dimensión de Empresas

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

| | Tabla |
|---|---|
| Entradas (solo Silver) | `datos_empresas.silver.de_registromercantil`, `datos_empresas.silver.de_actividadeseconomicas` |
| Salida | `datos_empresas.gold.de_dim_empresas` |
| Grano | Una fila por empresa: `llave_empresa` = `codigo_camara-matricula` |

**Depende de `03_Silver_RUES`** (que a su vez depende de `02_Silver_CIIU`).

## Propósito analítico

Es la dimensión que describe a cada empresa del registro mercantil, ya clasificada por su actividad económica. Responde:

* ¿Qué actividades económicas (CIIU) concentran más empresas activas y cuáles más cancelaciones?
* ¿Cómo se distribuyen las empresas por organización jurídica (SAS, fundaciones, persona natural…) y por cámara de comercio?
* ¿Cuántos años duran las empresas que se cancelan?

Además, `gold.de_fact_contratos` la usa para saber si el proveedor de un contrato está registrado en RUES.

## Transformaciones

| Columna Gold | Origen / cálculo | Justificación |
|---|---|---|
| `llave_empresa` | `codigo_camara` + `-` + `matricula` | Llave única y legible. La matrícula sola se repite entre cámaras. |
| `descripcion_ciiu_principal` / `_secundario` | `JOIN` con el catálogo CIIU de Silver | Traduce el código a una actividad legible. Se usa `LEFT JOIN` en el secundario porque es opcional. |
| `division_ciiu_principal` | 2 primeros dígitos del CIIU | Agrupa las ~500 clases en ~90 divisiones, más útiles para los gráficos. |
| `es_activa` / `es_cancelada` | `estado_matricula` y `fecha_cancelacion` | Indicadores booleanos listos para contar y filtrar. |
| `anio_matricula` | `YEAR(fecha_matricula)` | Permite agrupar por cohorte de creación. |
| `anios_hasta_cancelacion` | `(fecha_cancelacion - fecha_matricula) / 12 meses` | Vida de la empresa. Solo aplica a las canceladas; no usa `current_date`, así el resultado no cambia de un día a otro y el MERGE sigue siendo idempotente. |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Construir la dimensión desde Silver

In [ ]:
%python
rues = spark.table(SILVER_RUES).alias("r")
ciiu_pri = spark.table(SILVER_CIIU).select(F.col("codigo_ciiu").alias("ciiu_p"), F.col("descripcion_ciiu").alias("desc_p"))
ciiu_sec = spark.table(SILVER_CIIU).select(F.col("codigo_ciiu").alias("ciiu_s"), F.col("descripcion_ciiu").alias("desc_s"))

df_dim = (
    rues
    .join(F.broadcast(ciiu_pri), F.col("r.codigo_ciiu_principal") == F.col("ciiu_p"), "left")
    .join(F.broadcast(ciiu_sec), F.col("r.codigo_ciiu_secundario") == F.col("ciiu_s"), "left")
    .select(
        F.concat_ws("-", "r.codigo_camara", "r.matricula").alias("llave_empresa"),
        "r.codigo_camara", "r.camara_comercio", "r.matricula",
        "r.razon_social", "r.sigla",
        "r.clase_identificacion", "r.numero_identificacion_normalizado", "r.nit_normalizado",
        "r.organizacion_juridica", "r.tipo_sociedad", "r.categoria_matricula", "r.estado_matricula",
        (F.col("r.estado_matricula") == "ACTIVA").alias("es_activa"),
        (F.col("r.fecha_cancelacion").isNotNull()
         | F.coalesce(F.col("r.estado_matricula").contains("CANCELADA"), F.lit(False))).alias("es_cancelada"),
        "r.fecha_matricula",
        F.year("r.fecha_matricula").alias("anio_matricula"),
        "r.fecha_renovacion", "r.ultimo_ano_renovado", "r.fecha_cancelacion",
        F.round(F.months_between("r.fecha_cancelacion", "r.fecha_matricula") / 12, 1).cast("decimal(6,1)")
         .alias("anios_hasta_cancelacion"),
        "r.codigo_ciiu_principal",
        F.col("desc_p").alias("descripcion_ciiu_principal"),
        F.substring("r.codigo_ciiu_principal", 1, 2).alias("division_ciiu_principal"),
        "r.codigo_ciiu_secundario",
        F.col("desc_s").alias("descripcion_ciiu_secundario"),
    )
)

COLUMNAS_NEGOCIO = [c for c in df_dim.columns]
df_dim = df_dim.withColumn("_hash_registro", hash_columnas(COLUMNAS_NEGOCIO))

---
## Paso 2: Escritura idempotente (MERGE por `llave_empresa`)

In [ ]:
%python
m_gold = escribir_merge(
    df_dim, GOLD_DIM_EMPRESAS, ["llave_empresa"], capa="gold",
    comentario="Gold - Dimension de empresas del RUES con su actividad economica CIIU. Una fila por camara-matricula.",
)

n_silver, n_gold = spark.table(SILVER_RUES).count(), spark.table(GOLD_DIM_EMPRESAS).count()
assert n_silver == n_gold, f"La dimensión ({n_gold}) debe tener una fila por empresa de Silver ({n_silver})"
print(f"Empresas en Silver: {n_silver:,} | Empresas en la dimensión Gold: {n_gold:,}")

---
## Paso 3: Consultas de ejemplo

In [ ]:
%sql
-- Actividades económicas con más cancelaciones (tasa de cancelación)
SELECT division_ciiu_principal, FIRST(descripcion_ciiu_principal) AS ejemplo_actividad,
       COUNT(*) AS empresas,
       SUM(CASE WHEN es_cancelada THEN 1 ELSE 0 END) AS canceladas,
       ROUND(100 * AVG(CASE WHEN es_cancelada THEN 1 ELSE 0 END), 1) AS pct_canceladas
FROM datos_empresas.gold.de_dim_empresas
GROUP BY division_ciiu_principal
HAVING COUNT(*) >= 100
ORDER BY pct_canceladas DESC
LIMIT 10;

In [ ]:
%sql
-- Empresas por organización jurídica
SELECT organizacion_juridica, COUNT(*) AS empresas,
       SUM(CASE WHEN es_activa THEN 1 ELSE 0 END) AS activas
FROM datos_empresas.gold.de_dim_empresas
GROUP BY organizacion_juridica
ORDER BY empresas DESC
LIMIT 10;

In [ ]:
%python
finalizar({"notebook": "06_Gold_Dim_Empresas", "gold": m_gold})